[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phj9187/koisk-simulation/blob/codex/realism-20261030/notebooks/run_actor_critic_live.ipynb)

# 모델별 실시간 비교 · KREI 실제 매장 자료

파레토 2단계 탐색으로 추천 없음과 두 Actor–Critic AI를 비교합니다. 매장은 KREI 2025 외식업 경영실태 원자료에서 LHS로 뽑은 표본이고, 매장별 평균 방문자 수를 중심으로 날짜마다 방문자 수가 포아송 분포로 달라집니다. 서울시 추정매출 자료를 연결하면 실제 시간대별 매출 패턴으로 손님이 도착합니다. 세 방식의 결정시간 공식은 동일합니다. 기존 결과는 보존하고 새 실험 ID를 만듭니다. 결과·계획·체크포인트는 Drive에 저장합니다. 아래 실시간 카드에서 모델을 클릭하면 확대 그래프, α·β·λ, 실제 추천 가중치를 확인할 수 있습니다. 순위는 완료된 별도 동결 평가 구간의 미추천 대비 종합 효용 개선량입니다.


## 0. GitHub ↔ Colab 연동

이 노트북은 항상 위의 **Open in Colab** 배지로 여세요. 그러면 검증 브랜치 `codex/realism-20261030`의 노트북이 열립니다. Drive에 있는 옛 사본(`Kiosk_Actor_Critic_Live.ipynb` 등)은 더 이상 쓰지 않습니다.

노트북을 고쳤다면 **파일 → GitHub에 사본 저장**을 누르고 저장소 `phj9187/koisk-simulation`, 브랜치 `codex/realism-20261030`, 경로 `notebooks/run_actor_critic_live.ipynb`를 그대로 선택해 저장하세요. 처음 한 번은 GitHub 권한 승인 창이 뜹니다. 코드(`src/`, `config/`)는 아래 1번 셀이 실행할 때마다 GitHub에서 새로 받아옵니다.


## 1. GitHub 버전

공개 저장소 읽기에는 토큰이 필요 없습니다. 비공개 저장소만 Colab Secrets에 `KIOSK_GITHUB_TOKEN`을 등록하세요. 재개 시 `REVISION`에 progress.json의 git_commit을 입력하세요.

In [ ]:
import os, re, sys, subprocess, tempfile
from pathlib import Path
REPOSITORY = "phj9187/koisk-simulation" #@param {type:"string"}
REVISION = "codex/realism-20261030" #@param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("저장소는 owner/name 형식이어야 합니다.")
if not re.fullmatch(r"[A-Za-z0-9_./-]+", REVISION) or REVISION.startswith("-"):
    raise ValueError("브랜치 또는 커밋 해시를 입력하세요.")
if any(n.startswith(("kiosk_research", "kiosk_sim", "kiosk_cloud")) for n in sys.modules):
    raise RuntimeError("이전 코드가 로드되어 있습니다. 런타임을 다시 시작한 뒤 실행하세요.")
token = None
try:
    from google.colab import userdata
    token = userdata.get("KIOSK_GITHUB_TOKEN")
except Exception:
    pass
CODE_ROOT = Path(tempfile.mkdtemp(prefix="kiosk-research-", dir="/content"))
with tempfile.TemporaryDirectory() as auth_dir:
    helper = Path(auth_dir) / "askpass.py"
    helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('KIOSK_EPHEMERAL_TOKEN',''))\n")
    helper.chmod(0o700)
    child_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(helper))
    if token:
        child_env["KIOSK_EPHEMERAL_TOKEN"] = token
    try:
        commands = [
            ["git", "init", str(CODE_ROOT)],
            ["git", "-C", str(CODE_ROOT), "remote", "add", "origin", "https://github.com/"+REPOSITORY+".git"],
            ["git", "-C", str(CODE_ROOT), "fetch", "--depth", "1", "origin", REVISION],
            ["git", "-C", str(CODE_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        ]
        for command in commands:
            result = subprocess.run(command, env=child_env, capture_output=True, text=True)
            if result.returncode:
                raise RuntimeError("GitHub 읽기 실패. 저장소·버전·비공개 저장소 권한을 확인하세요. 토큰은 출력하지 않습니다.")
    finally:
        child_env.pop("KIOSK_EPHEMERAL_TOKEN", None)
        token = None
COMMIT = subprocess.check_output(["git","-C",str(CODE_ROOT),"rev-parse","HEAD"],text=True).strip()
subprocess.check_call([sys.executable,"-m","pip","install","-q","-r",str(CODE_ROOT/"requirements.txt")])
sys.path.insert(0,str(CODE_ROOT/"src"))
print("코드 경로:", CODE_ROOT)
print("실행 커밋:", COMMIT)


## 2. Drive 연결

처음 인증 요청은 직접 승인하세요. 공유 PC에서는 본인 계정인지 확인하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DATA_ROOT = Path("/content/drive/MyDrive/AI-Kiosk-Data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)


## 2-1. 공식 원자료 새로 확보

KREI 2025 원자료, 서울시 2025 추정매출과 생활인구, 미국 Census 외식업 매출·계절조정 계수를 공식 사이트에서 새로 확보합니다. 원본과 해시는 `raw_data/realism_20261030/sources.json`에 보존합니다. 기존 원자료·실험은 덮어쓰지 않습니다. 해외 지표는 별도의 foreign_stress 설정에서만 사용합니다.


In [ ]:
import runpy
USE_SALES_PROFILE = True #@param {type:"boolean"}
SALES_QUARTER = "" #@param {type:"string"}
RAW = DATA_ROOT / "raw_data" / "realism_20261030"
required = ["source_2025.xlsx", "seoul_sales.csv", "population_hourly.csv", "us_2025_monthly_scenario.csv", "sources.json"]
if not all((RAW / name).is_file() for name in required):
    downloader = runpy.run_path(str(CODE_ROOT / "tools" / "fetch_demand_data.py"))
    downloader["fetch"](RAW)
SURVEY = RAW / "source_2025.xlsx"
SALES = RAW / "seoul_sales.csv"
from kiosk_cloud.data import ensure_survey
ensure_survey(SURVEY)
print("공식 원자료와 해시 기록:", RAW / "sources.json")


## 3. 현실 조건 pilot

기본 realistic_pilot은 실제 매장 8곳, 5개 에피소드이며 각 에피소드는 3일입니다. 재고 노화·다음 날 입고·폐기·대기 포기·조리 변동·2개 추천을 비교합니다. 15개 달력 날짜를 사용하고 기존 하루 점수와 섞지 않습니다. foreign_stress는 미국 계절조정 계수를 수요의 별도 가정으로 적용합니다. 고객 반응 계수는 실측 보정값이 아닙니다. 먼저 pilot으로 실행시간과 진단을 확인하세요.


In [ ]:
import yaml
from datetime import datetime,timezone
from kiosk_research.config import Config
from kiosk_research.live import run_live
PRESET = "realistic_pilot" #@param ["realistic_pilot", "foreign_stress", "pilot", "research", "continual"]
DEADLINE_KST = "" #@param {type:"string"}
RESUME_ID = "" #@param {type:"string"}
deadline = datetime.fromisoformat(DEADLINE_KST) if DEADLINE_KST.strip() else None
if deadline is not None and (deadline.tzinfo is None or deadline <= datetime.now(timezone.utc)):
    raise ValueError("미래의 시간대 포함 중지 시각을 입력하세요.")
config = Config(**yaml.safe_load((CODE_ROOT / "config" / ("actor_critic_"+PRESET+".yaml")).read_text(encoding="utf-8")))
config.survey_path = str(SURVEY)
config.sales_profile_path = str(SALES) if USE_SALES_PROFILE else ""
config.sales_quarter = SALES_QUARTER.strip() if USE_SALES_PROFILE else ""
if config.realism.get("population_path"):
    config.realism["population_path"] = str(RAW / "population_hourly.csv")
if config.realism.get("monthly_profile_path"):
    config.realism["monthly_profile_path"] = str(RAW / "us_2025_monthly_scenario.csv")
config.validate()
if deadline is not None: config.early_stopping = False
config.auto_cleanup = False
assert 'no_recommendation' in config.models and config.decision_time_mode == 'equal'
assert config.lambda_search == 'pareto_two_stage'
run = run_live(DATA_ROOT, config, deadline.astimezone(timezone.utc).isoformat() if deadline else None,
               resume_id=RESUME_ID.strip() or None, target_day=None)
print("Saved:",run.id,run.state["day"],run.state["stop_reason"])


## 4. 진행 상황·저장 결과

이 셀은 실행 완료 후 자동으로 표시됩니다. 실행 중에는 Drive의 latest_run.json 또는 experiments/실험ID/progress.json에서 마지막 갱신 시각·완료 날짜·체크포인트·완료 매장 수를 확인하세요. 파일의 running 표시가 오래됐다면 현재 런타임이 살아 있다는 보장은 아닙니다.

In [ ]:
import json
from kiosk_research.session import status
print(json.dumps(status(DATA_ROOT), ensure_ascii=False, indent=2))
from kiosk_research.persistence import read_json
print(read_json(run.directory / "session_result.json"))


## 5. 선택: GitHub 요약 업로드

아래 설정을 true로 바꾼 경우에만 Colab Secrets의 쓰기 권한 토큰을 사용합니다. 원시 자료·체크포인트는 Drive에 남고 작은 요약만 새로운 results 브랜치에 올라갑니다. 이미 같은 요약을 올렸으면 저장된 업로드 기록을 보여줍니다.

In [ ]:
UPLOAD_SUMMARY = False #@param {type:"boolean"}
if UPLOAD_SUMMARY:
    from google.colab import userdata
    from kiosk_research.github import upload_summary
    summary = Path(read_json(run.directory / "session_result.json")["summary"])
    if (summary / "github_upload.json").exists():
        print(read_json(summary / "github_upload.json"))
    else:
        print(upload_summary(summary, REPOSITORY, userdata.get("KIOSK_GITHUB_TOKEN")))
